# Latency & Stress Tests — University Assistant
Measures response time per endpoint and simulates concurrent student sessions.

In [ ]:
import time
import requests
import concurrent.futures
import statistics
import uuid
import pandas as pd
import matplotlib.pyplot as plt

API_BASE = 'http://localhost:8000'

def timed_request(method, url, **kwargs):
    start = time.perf_counter()
    try:
        r = getattr(requests, method)(url, **kwargs, timeout=120)
        elapsed = time.perf_counter() - start
        return elapsed, r.status_code, len(r.content)
    except Exception as e:
        elapsed = time.perf_counter() - start
        return elapsed, 0, 0

print('Latency test setup complete.')

## 1. Single-endpoint Latency

In [ ]:
N_RUNS = 5

endpoint_tests = [
    ('Timetable', 'post', f'{API_BASE}/timetable', {'program': 'BTech', 'year': 2}),
    ('Course Lookup', 'get', f'{API_BASE}/course/AI101', {}),
    ('Fee Structure', 'get', f'{API_BASE}/fees/BTech', {}),
    ('Calendar', 'post', f'{API_BASE}/calendar', {'semester': 'odd', 'query_type': 'all'}),
    ('Faculty (CS dept)', 'post', f'{API_BASE}/faculty', {'department': 'Computer Science'}),
    ('RAG Retrieval', 'post', f'{API_BASE}/rag', {'query': 'What are prerequisites for DS201?', 'top_k': 5}),
]

latency_results = []

for name, method, url, payload in endpoint_tests:
    times = []
    for _ in range(N_RUNS):
        if method == 'post':
            t, status, size = timed_request('post', url, json=payload)
        else:
            t, status, size = timed_request('get', url)
        times.append(t)

    latency_results.append({
        'Endpoint': name,
        'Min (s)': round(min(times), 3),
        'Avg (s)': round(statistics.mean(times), 3),
        'Max (s)': round(max(times), 3),
        'Stdev': round(statistics.stdev(times) if len(times) > 1 else 0, 3),
        'Status': status,
    })
    print(f'  {name}: avg={statistics.mean(times):.3f}s')

df_lat = pd.DataFrame(latency_results)
print('\n', df_lat.to_string(index=False))

In [ ]:
# LLM chat latency (measured separately since it depends on Ollama)
print('Testing /chat endpoint latency...')
chat_times = []
for i in range(3):
    session = str(uuid.uuid4())
    t, status, _ = timed_request('post', f'{API_BASE}/chat',
        json={'session_id': session, 'message': 'What are the prerequisites for ML301?'})
    chat_times.append(t)
    print(f'  Run {i+1}: {t:.2f}s (status={status})')

print(f'Chat avg: {statistics.mean(chat_times):.2f}s')

## 2. Concurrent Load Test (200 simulated students)

In [ ]:
def student_request(student_id):
    session = f'stress_test_{student_id}'
    queries = [
        ('post', f'{API_BASE}/timetable', {'program': 'BTech', 'year': 1 + (student_id % 4)}),
        ('get', f'{API_BASE}/course/CS201', {}),
        ('post', f'{API_BASE}/rag', {'query': 'attendance policy', 'top_k': 3}),
    ]
    results = []
    for method, url, payload in queries:
        t, status, _ = timed_request(method, url, json=payload if method == 'post' else None)
        results.append((url.split('/')[-1], t, status))
    return results

N_STUDENTS = 50  # Reduce from 200 for local testing; raise on server
print(f'Running concurrent load test with {N_STUDENTS} virtual students...')

start_total = time.perf_counter()
all_results = []

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(student_request, i) for i in range(N_STUDENTS)]
    for f in concurrent.futures.as_completed(futures):
        all_results.extend(f.result())

total_time = time.perf_counter() - start_total
successful = sum(1 for _, _, s in all_results if s == 200)
failed = len(all_results) - successful

times_all = [t for _, t, _ in all_results]
print(f'\nTotal requests: {len(all_results)}')
print(f'Successful: {successful} | Failed: {failed}')
print(f'Total wall time: {total_time:.2f}s')
print(f'Throughput: {len(all_results)/total_time:.1f} req/s')
print(f'Avg latency: {statistics.mean(times_all):.3f}s | P95: {sorted(times_all)[int(len(times_all)*0.95)]:.3f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Endpoint latency bar chart
axes[0].barh(df_lat['Endpoint'], df_lat['Avg (s)'], color='steelblue', xerr=df_lat['Stdev'])
axes[0].set_xlabel('Average Response Time (s)')
axes[0].set_title('Endpoint Latency')
axes[0].axvline(1.0, color='red', linestyle='--', label='1s threshold')
axes[0].legend()

# Latency distribution histogram
axes[1].hist(times_all, bins=30, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Response Time (s)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Latency Distribution ({N_STUDENTS} concurrent students)')
axes[1].axvline(statistics.mean(times_all), color='red', linestyle='--', label='Mean')
axes[1].legend()

plt.tight_layout()
plt.savefig('latency_results.png', dpi=150)
plt.show()